In [1]:
import xml.etree.ElementTree as ET
import pandas as pd
from pathlib import Path

In [2]:
def parse_half_inning(half, inn_num, vh, game_meta):
    inn_summary = half.find('./innsummary')
    if inn_summary is None:
        return [], False

    try:
        total_runs = int(inn_summary.get('r', 0))
    except (ValueError, TypeError):
        return [], False

    rows = []
    runs_so_far = 0
    final_outs = 0

    for play in half.findall('./play'):
        outs_start = int(play.get('outs', 0))
        batter = play.find('./batter')
        is_pa = batter is not None and batter.get('action', '').strip() != ''

        if is_pa and outs_start < 3:
            b1 = 1 if 'first' in play.attrib else 0
            b2 = 1 if 'second' in play.attrib else 0
            b3 = 1 if 'third' in play.attrib else 0
            rows.append({
                **game_meta,
                'inning': int(inn_num),
                'half': vh,
                'start_outs': outs_start,
                'base_state': f"{b1}{b2}{b3}",
                'rei': total_runs - runs_so_far,
            })

        runs_so_far += len(play.findall('./*[@scored="1"]'))
        outs_made = sum(
            1 for el in list(play.findall('./batter')) + list(play.findall('./runner'))
            if el.get('out') == '1'
        )
        final_outs = outs_start + outs_made

    return rows, final_outs >= 3

In [3]:
def process_game(file_path, seen):
    try:
        tree = ET.parse(file_path)
    except ET.ParseError:
        return []

    root = tree.getroot()
    venue = root.find('./venue')
    if venue is None:
        return []

    date = venue.get('date', '')
    game_id = f"{date}_{venue.get('visid', '')}_{venue.get('homeid', '')}_{venue.get('start', '')}"

    if game_id in seen:
        return []
    seen.add(game_id)

    try:
        season = int(date.split('/')[-1])
    except (ValueError, IndexError):
        season = None

    game_meta = {
        'game_id': game_id,
        'date': date,
        'season': season,
        'vis_name': venue.get('visname', ''),
        'home_name': venue.get('homename', ''),
    }

    rows = []
    for inning in root.findall('.//inning'):
        inn_num = inning.get('number')
        for half in inning.findall('./batting'):
            half_rows, completed = parse_half_inning(half, inn_num, half.get('vh'), game_meta)
            if completed:
                rows.extend(half_rows)

    return rows

In [4]:
def parse_directory(base_dir, pattern='**/*.xml'):
    rows = []
    seen = set()
    for fp in Path(base_dir).glob(pattern):
        if fp.is_file():
            rows.extend(process_game(fp, seen))
    return pd.DataFrame(rows)


def build_matrix(df):
    m = df.groupby(['start_outs', 'base_state'])['rei'].agg(['mean', 'count']).reset_index()
    m['mean'] = m['mean'].round(3)
    return m.rename(columns={'mean': 'expected_runs'}).sort_values(['start_outs', 'base_state'])


def cornbelters_only(df):
    return df[df['vis_name'].str.contains('CornBelters', case=False, na=False) |
              df['home_name'].str.contains('CornBelters', case=False, na=False)]

In [5]:
if __name__ == '__main__':
    df = parse_directory('.')
    print(f"{len(df)} PAs across {df['game_id'].nunique()} games")

    out = Path('matrices')
    out.mkdir(exist_ok=True)

    build_matrix(df).to_csv(out / 'pl_all.csv', index=False)
    for yr in sorted(df['season'].dropna().unique()):
        build_matrix(df[df['season'] == yr]).to_csv(out / f'pl_{int(yr)}.csv', index=False)

    cb = cornbelters_only(df)
    build_matrix(cb).to_csv(out / 'cornbelters_all.csv', index=False)
    for yr in sorted(cb['season'].dropna().unique()):
        build_matrix(cb[cb['season'] == yr]).to_csv(out / f'cornbelters_{int(yr)}.csv', index=False)

    grid = build_matrix(df).pivot(index='base_state', columns='start_outs', values='expected_runs')
    print(grid)

153970 PAs across 1983 games


start_outs      0      1      2
base_state                     
000         0.759  0.384  0.139
001         1.698  1.137  0.434
010         1.495  0.875  0.388
011         2.379  1.606  0.710
100         1.240  0.702  0.286
101         2.163  1.383  0.633
110         1.913  1.194  0.540
111         2.946  1.955  0.955
